# Alquiler — 03 Feature engineering

Transforma los artefactos de `02_Limpieza.ipynb` (`rentals_train.csv` /
`rentals_test.csv`) en una matriz lista para la regresión lineal de
`log_rent`. Cada variable nueva nace de una hipótesis y se acepta o se
descarta con evidencia calculada en train; también quedan registrados los
descartes.

Este notebook **no** ajusta el modelo final ni hace selección de variables:
eso queda para la etapa siguiente. Sí deja planteado el costo de esa
selección, porque la cantidad final de predictores `p` determina si un
procedimiento exhaustivo es viable o no.

## 0. Entorno

Correr con el kernel **(.venv)**. Si falta el entorno:

```
source .venv/bin/activate
pip install -r requirements.txt
```

In [35]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

DATA_DIR = Path("..") / "data"
OUT_DIR = DATA_DIR / "processed"
TARGET = "monthly_rent_rub"
Y = "log_rent"
SEED = 42

# Orden real de la renovacion (de peor a mejor estado); se usa solo para
# poner a prueba la codificacion ordinal contra el one-hot, en la seccion 3.
RENOVATION_ORDER = {"no_renovation": 0, "cosmetic": 1, "euro": 2, "designer": 3}

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

print("pandas", pd.__version__, "| numpy", np.__version__)
print("DATA_DIR:", DATA_DIR.resolve(), "| existe:", DATA_DIR.exists())

pandas 3.0.6 | numpy 2.5.4
DATA_DIR: /Users/tomasmontilla/workspace/facultad/Moscow-real-estate-market--Sales-and-rentals/data | existe: True


### Contrato de entrada

Se hereda de `02_Limpieza.ipynb`, sin rehacerlo:

- El split 80/20 ya está hecho (16.000 train / 4.000 test, estratificado por
  `okrug`, semilla 42): acá no se vuelve a dividir.
- El target ya está en log (`log_rent`, skew 1,82 → −0,14) y se conserva el
  crudo `monthly_rent_rub` solo como referencia.
- `rent_per_sqm` ya salió por fuga (es el target dividido por la superficie)
  e `id` es solo trazabilidad: ninguno de los dos puede ser predictor.
- Ya existen `flag_invalido` y los flags de atípicos con umbrales de train.

**Frontera que rige todo el notebook:** lo derivable fila a fila (cocientes,
logaritmos, antigüedad) se calcula igual en train y en test; todo lo que se
*aprende* de los datos (medias y desvíos del escalado, categorías y bases del
one-hot, centros de los términos polinómicos) se ajusta solo en train y se
aplica a test. Si un parámetro pudiera salir distinto usando todo el
dataset, se usa el de train y se verifica.

In [36]:
train = pd.read_csv(OUT_DIR / "rentals_train.csv", parse_dates=["date_posted"])
test = pd.read_csv(OUT_DIR / "rentals_test.csv", parse_dates=["date_posted"])

assert train.shape == (16000, 42), f"train inesperado: {train.shape}"
assert test.shape == (4000, 42), f"test inesperado: {test.shape}"
assert set(train["id"]).isdisjoint(test["id"]), "solape de id entre train y test"
assert "rent_per_sqm" not in train.columns and "rent_per_sqm" not in test.columns
print(f"train {len(train):,} x {train.shape[1]} | test {len(test):,} x {test.shape[1]}")
print("sin solape de id y sin rent_per_sqm (fuga de 02): OK")
display(train[[Y, TARGET, "total_area", "rooms", "to_center_km"]].head(3))

train 16,000 x 42 | test 4,000 x 42
sin solape de id y sin rent_per_sqm (fuga de 02): OK


,log_rent,monthly_rent_rub,total_area,rooms,to_center_km
0,10.090009,24100,59.8,2,25.38
1,11.248973,76800,55.4,2,10.86
2,10.716660,45100,37.4,1,4.20


## 1. Transformaciones numéricas: log solo donde se justifica

Las tres continuas con cola derecha son `total_area`, `metro_distance_min` y
`to_center_km`. El logaritmo no se aplica por reflejo: tiene que achicar la
cola **y** mejorar la relación con `log_rent`. Se comparan tres formas —
cruda, `log1p` y raíz cuadrada — primero univariadas y después dentro de una
base numérica común, porque el ranking de una variable sola puede no
sostenerse cuando las demás ya están (las dos distancias comparten
información: su correlación cruda es 0,90).

Se usa `log1p` y no `log` porque hay ceros reales: la distancia al
metro llega a 0 minutos y la distancia al centro a 0,02 km.

In [37]:
FORMAS = {"cruda": lambda s: s.astype(float), "log1p": np.log1p, "sqrt": np.sqrt}
filas = []
for col in ["total_area", "metro_distance_min", "to_center_km"]:
    f = {"variable": col, "skew_cruda": stats.skew(train[col])}
    for nombre, fn in FORMAS.items():
        f[f"skew_{nombre}"] = stats.skew(fn(train[col]))
        f[f"r_{nombre}"] = fn(train[col]).corr(train[Y])
    filas.append(f)
display(pd.DataFrame(filas).round(3))

kf = KFold(5, shuffle=True, random_state=SEED)
def cv_r2(X):
    return cross_val_score(LinearRegression(), X, train[Y], cv=kf, scoring="r2").mean()

def base_num(area="log1p", centro="log1p", metro="log1p"):
    return pd.DataFrame({
        "area": FORMAS[area](train["total_area"]),
        "centro": FORMAS[centro](train["to_center_km"]),
        "metro": FORMAS[metro](train["metro_distance_min"]),
        "deposit": train["deposit_months"].astype(float),
    })

configs = [
    ("area log1p | centro log1p | metro log1p", base_num()),
    ("area log1p | centro sqrt | metro sqrt", base_num(centro="sqrt", metro="sqrt")),
    ("area cruda | centro cruda | metro cruda", base_num(area="cruda", centro="cruda", metro="cruda")),
    ("solo centro log1p (sin metro)", base_num().drop(columns=["metro"])),
    ("solo metro log1p (sin centro)", base_num().drop(columns=["centro"])),
]
tab_cv = pd.DataFrame({"configuracion": [c for c, _ in configs],
                       "CV_R2": [round(cv_r2(X), 4) for _, X in configs]})
display(tab_cv)
print(f"corr cruda metro-centro: {train['metro_distance_min'].corr(train['to_center_km']):.2f} | "
      f"en log1p: {np.log1p(train['metro_distance_min']).corr(np.log1p(train['to_center_km'])):.2f}")

,variable,skew_cruda,r_cruda,skew_log1p,r_log1p,skew_sqrt,r_sqrt
0,total_area,0.568,0.426,0.058,0.431,0.310,0.430
1,metro_distance_min,1.965,-0.674,-0.551,-0.640,0.766,-0.691
2,to_center_km,2.123,-0.695,-0.062,-0.712,1.051,-0.723


,configuracion,CV_R2
0,area log1p | centro log1p | metro log1p,0.7506
1,area log1p | centro sqrt | metro sqrt,0.7369
2,area cruda | centro cruda | metro cruda,0.6711
3,solo centro log1p (sin metro),0.6877
4,solo metro log1p (sin centro),0.5879


corr cruda metro-centro: 0.90 | en log1p: 0.62


In [38]:
for nombre, d in [("train", train), ("test", test)]:
    d["log_area"] = np.log1p(d["total_area"])
    d["log_center"] = np.log1p(d["to_center_km"])
    d["log_metro"] = np.log1p(d["metro_distance_min"])
print("creadas en train y test (fila a fila, sin parametros aprendidos): log_area, log_center, log_metro")
print(f"skew train: log_area {stats.skew(train['log_area']):.2f} | "
      f"log_center {stats.skew(train['log_center']):.2f} | log_metro {stats.skew(train['log_metro']):.2f}")

creadas en train y test (fila a fila, sin parametros aprendidos): log_area, log_center, log_metro
skew train: log_area 0.06 | log_center -0.06 | log_metro -0.55


**Decisiones de la sección 1.**

- `total_area` → `log_area`: el skew baja de 0,57 a 0,06 y es la forma que
  mejor rinde en la base conjunta. La ganancia univariada es chica y se
  declara como tal; alcanza con que no empeore y ordene la escala.
- `to_center_km` → `log_center`: skew 2,12 → −0,06, la mejora más clara de
  las tres. La relación por tramos ya se aplanaba en la periferia, que es
  exactamente lo que un log captura.
- `metro_distance_min` → `log_metro`: univariada, la raíz cuadrada
  correlaciona algo más (−0,69 contra −0,64 en log), pero en la base
  conjunta gana la combinación en log (ver CV de arriba) y la cruda es la
  peor de las tres formas. Se elige por el comportamiento en conjunto, no
  por el ranking de a una.
- Las dos distancias quedan por ahora: su correlación baja de 0,90 (crudas)
  al pasarlas a log y la colinealidad efectiva se mide con VIF en la
  sección 5 antes de cerrar la matriz. Dejar una sola es una decisión de
  selección, propia de la etapa de modelo.

## 2. Variables categóricas: one-hot, ordinal solo con justificación

OLS no consume texto y codificar etiquetas como 1, 2, 3 inventaría un orden
y distancias que no existen. El tratamiento por defecto es **one-hot con
k−1 columnas**: con constante en el modelo, la categoría que falta queda
como base y cada dummy mide la diferencia contra ella (con las k columnas
más la constante, una sobra por redundancia perfecta). La base elegida en
cada variable es su categoría mayoritaria en train, la más estable para
comparar.

El número como código solo se defiende si hay orden real **y** los saltos
entre niveles consecutivos son parecidos; si los saltos son desiguales, el
código obliga al modelo a tratarlos como iguales y miente. Por eso
`renovation` (que sí tiene orden: sin renovar < cosmética < euro <
designer) se pone a prueba en las dos codificaciones, y `rooms` también.

**Alta cardinalidad, afuera de la base:** `district` (129 niveles) y
`metro_station` (104) sumarían 128 + 103 = 231 dummies. La matriz pasaría
de unos 40 predictores a unos 270, y un procedimiento exhaustivo de
selección debería probar 2^p modelos: con p = 40 ya son del orden de 10^12.
No es un problema de filas (cada distrito tiene más de 100 avisos), es un
problema de dimensionalidad. La geografía fina ya entra, en versión
gruesa, por `okrug` y `metro_line`.

In [39]:
for col in ["rooms", "renovation"]:
    t = train.groupby(col, observed=True)[Y].agg(["mean", "count"])
    if col == "renovation":  # el orden que importa es el real, no el alfabetico
        t = t.reindex(["no_renovation", "cosmetic", "euro", "designer"])
    t["salto_vs_anterior"] = t["mean"].diff()
    print(f"--- {col} (medias de log_rent en train; saltos entre niveles) ---")
    display(t.round(3))
print("rooms: saltos decrecientes -> el efecto del cuarto ambiente no es el del segundo.")
print("renovation: orden real, pero saltos crecientes -> la igualdad de pasos tampoco se sostiene.")

CATS = ["rooms", "building_type", "renovation", "okrug", "metro_line"]
bases = {c: train[c].mode().iloc[0] for c in CATS}
print("base (mayoritaria en train) por variable:", bases)

enc = OneHotEncoder(handle_unknown="ignore", drop=[bases[c] for c in CATS],
                    sparse_output=False)
dum_train = pd.DataFrame(enc.fit_transform(train[CATS]),
                         columns=enc.get_feature_names_out(CATS), index=train.index)
dum_test = pd.DataFrame(enc.transform(test[CATS]),
                        columns=enc.get_feature_names_out(CATS), index=test.index)
print(f"one-hot (k-1) ajustado en train: {dum_train.shape[1]} columnas desde {CATS}")
grupos = {c: [n for n in dum_train.columns if n.startswith(c + "_")] for c in CATS}
for c, cols in grupos.items():
    s_tr = dum_train[cols].sum(axis=1)
    assert set(s_tr.unique()) <= {0.0, 1.0}, c
    print(f"{c}: {len(cols)} dummies | suma por fila en {{0,1}}: OK (0 = base '{bases[c]}')")
for c in CATS:
    nuevas = sorted(set(test[c].unique()) - set(train[c].unique()))
    print(f"categorias en test no vistas en train [{c}]: {nuevas if nuevas else 'ninguna'}")
print(f"si district y metro_station entraran: +{train['district'].nunique() - 1 + train['metro_station'].nunique() - 1}"
      f" columnas -> p pasaria de ~38 a ~{38 + train['district'].nunique() - 1 + train['metro_station'].nunique() - 1}")

--- rooms (medias de log_rent en train; saltos entre niveles) ---


,mean,count,salto_vs_anterior
rooms,,,
1,10.525,7230,NaN
2,10.801,5608,0.276
3,10.994,2408,0.194
4,11.154,754,0.160


--- renovation (medias de log_rent en train; saltos entre niveles) ---


,mean,count,salto_vs_anterior
renovation,,,
no_renovation,10.549,1582,NaN
cosmetic,10.650,7232,0.101
euro,10.793,5551,0.143
designer,10.963,1635,0.170


rooms: saltos decrecientes -> el efecto del cuarto ambiente no es el del segundo.
renovation: orden real, pero saltos crecientes -> la igualdad de pasos tampoco se sostiene.
base (mayoritaria en train) por variable: {'rooms': np.int64(1), 'building_type': 'panel', 'renovation': 'cosmetic', 'okrug': 'SVAO', 'metro_line': 'Kaluzhsko-Rizhskaya'}
one-hot (k-1) ajustado en train: 32 columnas desde ['rooms', 'building_type', 'renovation', 'okrug', 'metro_line']
rooms: 3 dummies | suma por fila en {0,1}: OK (0 = base '1')
building_type: 5 dummies | suma por fila en {0,1}: OK (0 = base 'panel')
renovation: 3 dummies | suma por fila en {0,1}: OK (0 = base 'cosmetic')
okrug: 10 dummies | suma por fila en {0,1}: OK (0 = base 'SVAO')
metro_line: 11 dummies | suma por fila en {0,1}: OK (0 = base 'Kaluzhsko-Rizhskaya')
categorias en test no vistas en train [rooms]: ninguna
categorias en test no vistas en train [building_type]: ninguna
categorias en test no vistas en train [renovation]: ninguna
categ

In [40]:
def matriz_cv(renovacion="onehot", rooms="dummies"):
    X = pd.DataFrame({
        "log_area": train["log_area"], "log_center": train["log_center"],
        "log_metro": train["log_metro"],
        "deposit": train["deposit_months"].astype(float),
        "furnished": train["furnished"].astype(float),
        "pets": train["pets_allowed"].astype(float),
    }, index=train.index)
    if renovacion == "ordinal":
        X["ren_ord"] = train["renovation"].map(RENOVATION_ORDER).astype(float)
    else:
        X = pd.concat([X, dum_train[[c for c in dum_train.columns if c.startswith("renovation_")]]], axis=1)
    if rooms == "num":
        X["rooms_num"] = train["rooms"].astype(float)
    else:
        X = pd.concat([X, dum_train[[c for c in dum_train.columns if c.startswith("rooms_")]]], axis=1)
    resto = [c for c in dum_train.columns
             if not c.startswith(("renovation_", "rooms_"))]
    return pd.concat([X, dum_train[resto]], axis=1)

comp = pd.DataFrame({
    "codificacion": ["renovation one-hot + rooms dummies",
                     "renovation ordinal + rooms dummies",
                     "renovation one-hot + rooms numerico"],
    "p": [matriz_cv().shape[1], matriz_cv("ordinal").shape[1], matriz_cv(rooms="num").shape[1]],
    "CV_R2": [round(cv_r2(matriz_cv()), 4),
              round(cv_r2(matriz_cv("ordinal")), 4),
              round(cv_r2(matriz_cv(rooms="num")), 4)],
})
display(comp)
print("deposit_months queda numerica (0-2): sus medias por nivel son casi planas en el EDA;")
print("se conserva como control y la etapa de modelo puede descartarla sin costo de codificacion.")

,codificacion,p,CV_R2
0,renovation one-hot + rooms dummies,38,0.9252
1,renovation ordinal + rooms dummies,36,0.9247
2,renovation one-hot + rooms numerico,36,0.9253


deposit_months queda numerica (0-2): sus medias por nivel son casi planas en el EDA;
se conserva como control y la etapa de modelo puede descartarla sin costo de codificacion.


**Decisiones de la sección 2.**

- `building_type`, `okrug` y `metro_line` (nominales): one-hot k−1 con base
  mayoritaria de train, 5 + 10 + 11 columnas. Sin orden que justificar, no
  hay alternativa numérica defendible.
- `renovation`: tiene orden, pero los saltos entre medias consecutivas no
  son iguales (crecen de cosmetic a designer), así que el código 0–3
  impondría pasos parejos que los datos no muestran. La comparación por CV
  de arriba empata o favorece levemente al one-hot a costa de 2 columnas
  más; se queda el **one-hot** (3 dummies), que no asume espaciamiento.
- `rooms`: los saltos decrecen (el paso de 1 a 2 ambientes vale más que el
  de 3 a 4), el mismo argumento. Se queda en **dummies** (3 columnas, base
  1 ambiente); la versión numérica rinde igual en CV, pero fuerza una
  recta sobre un efecto que se achica.
- `district` y `metro_station` quedan fuera de la matriz base por
  dimensionalidad (231 columnas extra) y constan como decisión, no como
  olvido: `okrug` y `metro_line` ya aportan la geografía en grueso.

## 3. Escalado: para comparar coeficientes y para seleccionar

Estandarizar (restar la media y dividir por el desvío de train) no cambia
el ajuste de un OLS sin penalizar: los mismos pronósticos salen con
coeficientes reescalados. Su utilidad acá es otra y se declara como tal:

- pone las continuas en una escala común, así los coeficientes se pueden
  comparar entre sí ("cuánto mueve al alquiler un desvío más de
  superficie" contra "un desvío más de distancia");
- es requisito práctico para que la selección y la regularización de la
  etapa siguiente traten a todas las variables con la misma vara (una
  variable en kilómetros y otra en minutos no son comparables en crudo).

Se escalan solo las continuas que entran al modelo. Las dummies y las
binarias ya viven en {0, 1} y escalarlas les rompería la lectura; `deposit_months` (0–2) queda en su escala por
su rango corto y comparable. El target no se toca: ya está en log.


In [41]:
CONT_Z = ["log_area", "log_center", "log_metro"]
scaler = StandardScaler().fit(train[CONT_Z])
z_train = pd.DataFrame(scaler.transform(train[CONT_Z]),
                       columns=[c + "_z" for c in CONT_Z], index=train.index)
z_test = pd.DataFrame(scaler.transform(test[CONT_Z]),
                      columns=[c + "_z" for c in CONT_Z], index=test.index)

params = pd.DataFrame({"media_train (usada)": scaler.mean_,
                       "desvio_train (usado)": scaler.scale_,
                       "media_global (prohibida)": pd.concat([train, test])[CONT_Z].mean().to_numpy()},
                      index=CONT_Z)
display(params.round(3))
assert np.allclose(scaler.mean_, train[CONT_Z].mean().to_numpy())
assert not np.allclose(scaler.mean_, pd.concat([train, test])[CONT_Z].mean().to_numpy())
print("OK: el escalado aprendio de train (16.000), no del dataset completo.")
print("media de cada z en train (debe ser ~0):", np.round(z_train.mean().to_numpy(), 6))
print("desvio de cada z en train (debe ser ~1):", np.round(z_train.std(ddof=0).to_numpy(), 6))
print("media de cada z en test (no tiene por que ser 0):", np.round(z_test.mean().to_numpy(), 3))

,media_train (usada),desvio_train (usado),media_global (prohibida)
log_area,4.007,0.223,4.007
log_center,2.640,0.652,2.641
log_metro,4.394,1.282,4.390


OK: el escalado aprendio de train (16.000), no del dataset completo.
media de cada z en train (debe ser ~0): [-0. -0.  0.]
desvio de cada z en train (debe ser ~1): [1. 1. 1.]
media de cada z en test (no tiene por que ser 0): [ 0.013  0.005 -0.015]


## 4. Matriz final, colinealidad y guardado

La matriz base reúne todo lo que sobrevivió a las secciones anteriores:

- **Continuas estandarizadas (parámetros de train):** `log_area`,
  `log_center` y `log_metro`, como columnas `_z`.
- **Sin escalar:** `deposit_months`, `furnished` y `pets_allowed`, ya en
  escalas cortas y comparables.
- **Dummies k−1 de la sección 2** (32 columnas).

Antes de guardar se chequea lo que puede romper a la etapa siguiente:
nulos, colinealidad (VIF: cuánto infla la varianza de cada coeficiente la
relación con las demás) y el tamaño del problema de selección que se le
entrega al modelo. El ajuste OLS que se imprime es solo una verificación
de sanidad de la matriz construida, no el modelo final: la selección entre
estas columnas — incluida la posibilidad de dejar una sola de las dos
distancias — es trabajo de la etapa siguiente.

Al guardar, `rental_price_per_sqm_monthly` sale de los artefactos de esta
etapa: nunca fue predictora (no está en `FEATURES_BASE`), pero viajaba
heredada de los archivos de 02. Queda fuera de la base con el mismo
criterio con que 02 sacó a `rent_per_sqm`: es un agregado distrital del
propio alquiler (r = 0,78 con `log_rent`), demasiado cerca del target
para convivir con el resto de las columnas.

In [42]:
ZCOLS = ["log_area_z", "log_center_z", "log_metro_z"]
DUMMIES = list(dum_train.columns)
FEATURES_BASE = ZCOLS + ["deposit_months", "furnished", "pets_allowed"] + DUMMIES

fin_train = pd.concat([z_train, train[["deposit_months",
                                       "furnished", "pets_allowed"]], dum_train], axis=1)[FEATURES_BASE]
fin_test = pd.concat([z_test, test[["deposit_months",
                                    "furnished", "pets_allowed"]], dum_test], axis=1)[FEATURES_BASE]
assert list(fin_train.columns) == FEATURES_BASE and list(fin_test.columns) == FEATURES_BASE
assert fin_train.shape == (16000, len(FEATURES_BASE)), fin_train.shape
assert fin_test.shape == (4000, len(FEATURES_BASE)), fin_test.shape
assert fin_train.isna().sum().sum() == 0 and fin_test.isna().sum().sum() == 0
assert len(FEATURES_BASE) == 38 and ZCOLS == ["log_area_z", "log_center_z", "log_metro_z"]
assert "id" not in FEATURES_BASE and TARGET not in FEATURES_BASE and Y not in FEATURES_BASE
print(f"matriz final: train {fin_train.shape} | test {fin_test.shape} | p = {len(FEATURES_BASE)} predictores")
print(f"Best Subset exhaustivo sobre p = {len(FEATURES_BASE)}: 2^{len(FEATURES_BASE)}"
      f" = {2 ** len(FEATURES_BASE):.3e} modelos -> inviable; la etapa de modelo debe acotar el pool")
print(f"con district + metro_station seria p = {len(FEATURES_BASE) + 231} (2^{len(FEATURES_BASE) + 231} modelos)")

M = sm.add_constant(fin_train, has_constant="add")
vifs = pd.Series([variance_inflation_factor(M.to_numpy(), i) for i in range(M.shape[1])],
                 index=M.columns).drop("const").sort_values(ascending=False)
print("VIF mas altos de la matriz final (train):")
display(vifs.head(8).round(2).rename("VIF").to_frame())
print(f"VIF maximo: {vifs.max():.2f} | predictores con VIF > 10: {int((vifs > 10).sum())}"
      " (el chequeo fino de colinealidad entre distancias queda para la seleccion del modelo)")

sane = sm.OLS(train[Y], sm.add_constant(fin_train, has_constant="add")).fit()
print(f"sanidad (no es el modelo final): adj_R2 en train = {sane.rsquared_adj:.4f} | "
      f"CV 5-fold R2 = {cv_r2(fin_train):.4f}")

matriz final: train (16000, 38) | test (4000, 38) | p = 38 predictores
Best Subset exhaustivo sobre p = 38: 2^38 = 2.749e+11 modelos -> inviable; la etapa de modelo debe acotar el pool
con district + metro_station seria p = 269 (2^269 modelos)
VIF mas altos de la matriz final (train):


,VIF
log_center_z,4.43
log_area_z,3.84
rooms_3,3.17
okrug_TAO,2.89
rooms_4,2.38
okrug_CAO,2.31
rooms_2,2.29
okrug_NAO,1.94


VIF maximo: 4.43 | predictores con VIF > 10: 0 (el chequeo fino de colinealidad entre distancias queda para la seleccion del modelo)
sanidad (no es el modelo final): adj_R2 en train = 0.9255 | CV 5-fold R2 = 0.9252


In [43]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
# rental_price_per_sqm_monthly sale de la base en esta etapa (ver seccion 4):
# heredada de 02, cercana al target y nunca predictora en FEATURES_BASE.
train = train.drop(columns=["rental_price_per_sqm_monthly"])
test = test.drop(columns=["rental_price_per_sqm_monthly"])
# Construccion explicita: original de 02 + logs, columnas z y dummies.
train_fe = pd.concat([train.drop(columns=[c for c in ZCOLS + DUMMIES if c in train.columns]),
                      z_train, dum_train], axis=1)
test_fe = pd.concat([test.drop(columns=[c for c in ZCOLS + DUMMIES if c in test.columns]),
                     z_test, dum_test], axis=1)
train_fe.to_csv(OUT_DIR / "rentals_train_fe.csv", index=False)
test_fe.to_csv(OUT_DIR / "rentals_test_fe.csv", index=False)
for f in ["rentals_train_fe.csv", "rentals_test_fe.csv"]:
    d = pd.read_csv(OUT_DIR / f)
    assert "rent_per_sqm" not in d.columns and "rental_price_per_sqm_monthly" not in d.columns
    print(f"{f}: {d.shape[0]:,} x {d.shape[1]} | matriz p={len(FEATURES_BASE)} embebida | "
          "sin rent_per_sqm ni rental_price_per_sqm_monthly")
print("FEATURES_BASE (en orden):", FEATURES_BASE)

rentals_train_fe.csv: 16,000 x 79 | matriz p=38 embebida | sin rent_per_sqm ni rental_price_per_sqm_monthly
rentals_test_fe.csv: 4,000 x 79 | matriz p=38 embebida | sin rent_per_sqm ni rental_price_per_sqm_monthly
FEATURES_BASE (en orden): ['log_area_z', 'log_center_z', 'log_metro_z', 'deposit_months', 'furnished', 'pets_allowed', 'rooms_2', 'rooms_3', 'rooms_4', 'building_type_brick', 'building_type_khrushchev', 'building_type_modern_panel', 'building_type_monolith', 'building_type_stalin', 'renovation_designer', 'renovation_euro', 'renovation_no_renovation', 'okrug_CAO', 'okrug_NAO', 'okrug_SAO', 'okrug_SZAO', 'okrug_TAO', 'okrug_VAO', 'okrug_YuAO', 'okrug_YuZAO', 'okrug_ZAO', 'okrug_ZelAO', 'metro_line_Arbatsko-Pokrovskaya', 'metro_line_Bolshaya Koltsevaya', 'metro_line_Filyovskaya', 'metro_line_Kalininskaya', 'metro_line_Koltsevaya', 'metro_line_Lyublinsko-Dmitrovskaya', 'metro_line_MCC', 'metro_line_Serpukhovsko-Timiryazevskaya', 'metro_line_Sokolnicheskaya', 'metro_line_Tagansko-

## 5. Resumen de transformaciones

| Variable | Resolución | Motivo |
|---|---|---|
| `monthly_rent_rub` | target en log (`log_rent`, heredado de 02) | skew 1,82 → −0,14 |
| `total_area` | `log_area`, estandarizada | cola derecha; mejor forma en conjunto |
| `to_center_km` | `log_center`, estandarizada | skew 2,12 → −0,06; periferia se aplana |
| `metro_distance_min` | `log_metro`, estandarizada | gana en conjunto; la cruda es la peor forma |
| `rooms` | dummies k−1 (base 1) | saltos entre niveles desiguales y decrecientes |
| `renovation` | one-hot k−1 (base `cosmetic`) | orden real pero pasos no iguales; CV no premia el ordinal |
| `building_type`, `okrug`, `metro_line` | one-hot k−1 (base mayoritaria en train) | nominales, sin orden defendible |
| `deposit_months` | numérica 0–2 sin escalar | rango corto; señal plana, queda como control |
| `furnished`, `pets_allowed` | binarias 0/1 (heredadas) | ya comparables |
| `district`, `metro_station` | **fuera de la base** | +231 dummies: p 38 → 269, selección exhaustiva inviable |
| `lat`, `lon`, coordenadas de la estación | **fuera** | redundantes con la distancia al centro; no aportan forma nueva |
| `rental_price_per_sqm_monthly` | **fuera de la base (columna eliminada)** | agregado distrital del propio alquiler (r = 0,78): sale de los artefactos FE como `rent_per_sqm` en 02 |
| `rent_per_sqm`, `id` | **nunca predictores** | fuga directa / solo trazabilidad |

**Qué se le entrega a la siguiente etapa:** train y test en
`data/processed/rentals_train_fe.csv` y `rentals_test_fe.csv`, con la lista
`FEATURES_BASE` (p = 38: 3 continuas estandarizadas, 3 numéricas/binarias
sin escalar y 32 dummies). Con ese p, Best Subset exhaustivo (2^38 ≈ 3×10^11
modelos) no es viable: la selección deberá acotar el pool candidato o usar
procedimientos secuenciales o regularizados, y decidir allí si conviven
las dos distancias en log o queda una sola. Los archivos FE van sin
`rent_per_sqm` (fuera desde 02) y sin `rental_price_per_sqm_monthly`
(eliminada en esta etapa); los artefactos de 02 no se tocaron.